# Test data from your SQL schema

Send your `CREATE TABLE` statements and get data that fits them: the same tables, columns and types, primary and foreign keys that match, `UNIQUE`, `NOT NULL` and `CHECK` respected. Add rules in plain English for what the schema cannot say.

**Uses:** 5,000 records, no AI-generated fields. API reference: [generaterealdata.com/docs/api/](https://generaterealdata.com/docs/api/).

## Setup

You need an API key (PRO, ULTRA or Enterprise plan): sign in at [generaterealdata.com](https://generaterealdata.com), open the dashboard and click **Generate API key**. Keep it out of the notebook: put it in the `GRD_API_KEY` environment variable, or in Colab's **Secrets** panel under the same name.

In [1]:
import os
import time
from pathlib import Path

import requests

API = "https://generaterealdata.com/api/v1"
KEY = os.environ.get("GRD_API_KEY")
if not KEY:                                   # Google Colab: read it from the Secrets panel
    from google.colab import userdata
    KEY = userdata.get("GRD_API_KEY")
HEADERS = {"Authorization": f"Bearer {KEY}"}
OUT = Path("downloads")
OUT.mkdir(exist_ok=True)


def api(method, path, **kwargs):
    """One API call. Waits and tries again if the API says to slow down or another dataset is still running."""
    while True:
        r = requests.request(method, API + path, headers=HEADERS, timeout=90, **kwargs)
        busy = r.status_code == 409 and r.json().get("error", {}).get("code") == "run_in_progress"
        if r.status_code == 429 or busy:
            time.sleep(int(r.headers.get("Retry-After", 15)))
            continue
        data = r.json()
        if r.status_code >= 400:
            e = data["error"]
            raise RuntimeError(f"{e['code']}: {e['message']} {e.get('hint', '')}")
        return data


def generate(spec):
    """Start a dataset and wait until it is ready. Returns its final status."""
    ds = api("POST", "/datasets", json=spec)
    while ds["status"] not in ("ready", "failed"):
        time.sleep(5)
        ds = api("GET", f"/datasets/{ds['id']}")
    if ds["status"] == "failed":
        raise RuntimeError(ds["error"]["message"])
    return ds


def download(ds, filename, file=None):
    """Save a ready dataset's file. file="db" fetches the SQLite file of an SQL dataset."""
    path = OUT / filename
    url = f"{API}/datasets/{ds['id']}/download"
    with requests.get(url, params={"file": file} if file else None, headers=HEADERS, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
    return path


def summary(ds):
    """The facts worth reading from a dataset's status."""
    keys = ("status", "format", "rows_generated", "rows_per_table", "credits_charged", "ai_rows_charged", "schema_id", "seed")
    return {k: ds[k] for k in keys if k in ds}

## 1. The schema and the rules

The example schema is a small shop with four tables, kept in [`schemas/shop.sql`](../schemas/shop.sql). The rules are in [`schemas/shop_rules.txt`](../schemas/shop_rules.txt).

In [2]:
def repo_file(path):
    """Read a file of this repository: from disk when the repo is cloned, from GitHub otherwise (e.g. in Colab)."""
    local = Path("..") / path
    if local.exists():
        return local.read_text(encoding="utf-8")
    r = requests.get("https://raw.githubusercontent.com/generaterealdata/api-examples/main/" + path, timeout=30)
    r.raise_for_status()
    return r.text


SHOP_SQL = repo_file("schemas/shop.sql")
SHOP_RULES = repo_file("schemas/shop_rules.txt").strip()
print(SHOP_SQL)

-- A small online shop: four related tables.
-- Used by the example notebooks. PostgreSQL syntax; structure only.

CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    full_name   VARCHAR(120) NOT NULL,
    email       VARCHAR(160) NOT NULL UNIQUE,
    country     VARCHAR(60)  NOT NULL,
    signup_date DATE         NOT NULL
);

CREATE TABLE products (
    product_id   INTEGER PRIMARY KEY,
    product_name VARCHAR(120)  NOT NULL,
    category     VARCHAR(20)   NOT NULL CHECK (category IN ('Electronics', 'Home', 'Kitchen', 'Toys', 'Sports', 'Books')),
    unit_price   NUMERIC(10,2) NOT NULL CHECK (unit_price > 0)
);

CREATE TABLE orders (
    order_id    INTEGER PRIMARY KEY,
    customer_id INTEGER       NOT NULL REFERENCES customers(customer_id),
    order_date  DATE          NOT NULL,
    status      VARCHAR(12)   NOT NULL CHECK (status IN ('pending', 'shipped', 'delivered', 'cancelled')),
    order_total NUMERIC(12,2) NOT NULL
);

CREATE TABLE order_items (
    order_item

In [3]:
print(SHOP_RULES)

An online shop. CUSTOMERS: country is one of United States (40%), United Kingdom (15%), Canada (12%), Germany (12%), France (11%), Australia (10%); signup_date between 2024-06-01 and 2025-12-31. PRODUCTS: product_name must read like a real product that fits its category, never a person's name or random words: build it with concat from a style word (Compact, Wireless, Classic, Deluxe, Eco, Everyday, Premium, Travel) and a product type chosen by conditional rules on category (Electronics: Headphones, Bluetooth Speaker, USB-C Charger, Webcam; Home: Table Lamp, Throw Blanket, Wall Clock; Kitchen: Kettle, Frying Pan, Knife Set, Blender; Toys: Board Game, Building Blocks, Plush Bear; Sports: Yoga Mat, Jump Rope, Water Bottle; Books: Cookbook, Puzzle Book, Travel Guide). unit_price between 5 and 400, mostly under 80. ORDERS: every customer has at least one order (at_least_one). order_date between 2025-01-01 and 2026-09-30 and never before the customer's signup_date (keep it inside the period)

## 2. Generate

`format: "sql"` returns a script for your database (`dialect` picks which one) and, as a second file, the same data as a ready-to-open SQLite database.

In [4]:
ds = generate({"mode": "sql", "sql": SHOP_SQL, "notes": SHOP_RULES, "rows": 5000,
               "format": "sql", "dialect": "postgresql", "seed": 2, "name": "Shop example"})
summary(ds)

{'status': 'ready',
 'format': 'sql',
 'rows_generated': 5000,
 'rows_per_table': {'order_items': 2950,
  'orders': 1500,
  'products': 50,
  'customers': 500},
 'credits_charged': 5000,
 'ai_rows_charged': 0,
 'schema_id': 'sch_c_9sntInhUY0VKvq',
 'seed': 2}

## 3. The SQL script

One file creates the tables and fills them in foreign-key order, inside a single transaction.

In [5]:
script = download(ds, "shop_postgresql.sql")
lines = script.read_text(encoding="utf-8").splitlines()
print("\n".join(lines[:16]))
print("...")
first_insert = next(i for i, line in enumerate(lines) if line.startswith("INSERT"))
print("\n".join(line[:150] for line in lines[first_insert:first_insert + 4]))

-- ============================================================
-- Synthetic dataset : Shop example
-- Generated         : 2026-10-11 16:02:28 UTC
-- Dialect           : PostgreSQL
-- Contents          : 4 table(s), 5,000 row(s)
-- Load with         : psql -d your_database -f this_file.sql
--
-- One database in one script. Tables are created and filled in
-- foreign-key order inside a single transaction.
--   customers                           500 rows
--   products                             50 rows
--   orders                            1,500 rows
--   order_items                       2,950 rows
-- ============================================================

BEGIN;
...
INSERT INTO customers ("customer_id", "full_name", "email", "country", "signup_date") VALUES
    (1, 'Theresa Brown', 'kelly58@example.net', 'Australia', '2025-03-01'),
    (2, 'Russell Fitzgerald', 'yreynolds@example.org', 'Australia', '2025-05-25'),
    (3, 'Elizabeth Obrien', 'hmiller@example.org', 'United State

Load it with `psql -d your_database -f shop_postgresql.sql`.

## 4. Query it right away

The SQLite file needs no server.

In [6]:
import sqlite3

import pandas as pd

con = sqlite3.connect(download(ds, "shop.sqlite", file="db"))
pd.read_sql_query("""
    SELECT c.full_name, c.country, COUNT(*) AS orders, ROUND(SUM(o.order_total), 2) AS spend
    FROM orders o JOIN customers c USING (customer_id)
    WHERE o.status <> 'cancelled'
    GROUP BY c.customer_id ORDER BY spend DESC LIMIT 5""", con)

,full_name,country,orders,spend
0,Samuel Curry,Germany,31,11012.06
1,Richard Martin,United States,15,8003.89
2,Jeffrey Cross,Germany,24,7835.78
3,Kelli Gonzales,United Kingdom,17,6359.21
4,Brenda Salazar,United States,11,5779.72


In [7]:
pd.read_sql_query("""
    SELECT p.category, COUNT(*) AS lines, SUM(i.quantity) AS units, ROUND(SUM(i.line_total), 2) AS revenue
    FROM order_items i JOIN products p USING (product_id)
    GROUP BY p.category ORDER BY revenue DESC""", con)

,category,lines,units,revenue
0,Electronics,1173,3527,157329.84
1,Kitchen,360,1061,105306.26
2,Toys,488,1491,101676.50
3,Books,383,1150,79267.40
4,Home,199,607,68773.60
5,Sports,347,1034,30045.75


## 5. The rules held

The rules asked for prices copied from the product, line totals of quantity x price and order totals that are the sum of their lines. Each query counts the rows that break a rule.

In [8]:
rules = {
    "item price differs from the product's price":
        "SELECT COUNT(*) FROM order_items i JOIN products p USING (product_id) WHERE ABS(i.unit_price - p.unit_price) > 0.005",
    "line_total is not quantity x unit_price":
        "SELECT COUNT(*) FROM order_items WHERE ABS(line_total - quantity * unit_price) > 0.011",
    "order_total is not the sum of its lines":
        "SELECT COUNT(*) FROM orders o JOIN (SELECT order_id, SUM(line_total) s FROM order_items GROUP BY order_id) x "
        "USING (order_id) WHERE ABS(o.order_total - x.s) > 0.011",
    "order placed before the customer signed up":
        "SELECT COUNT(*) FROM orders o JOIN customers c USING (customer_id) WHERE o.order_date < c.signup_date",
}
pd.DataFrame([(rule, con.execute(sql).fetchone()[0]) for rule, sql in rules.items()], columns=["rule", "rows breaking it"])

,rule,rows breaking it
0,item price differs from the product's price,0
1,line_total is not quantity x unit_price,0
2,order_total is not the sum of its lines,0
3,order placed before the customer signed up,0


## Next

- The same data for MySQL, SQL Server or SQLite: [notebook 04](04_load_into_postgres_mysql_sqlite.ipynb).
- A full set of checks you can reuse on any dataset: [notebook 06](06_check_the_data_foreign_keys_totals_dates.ipynb).
- More ready-made schemas are in [`schemas/`](../schemas/).